# Giai đoạn 3: Huấn luyện Kiến trúc YOLOv11 (BDD100k)
Tài liệu này thực thi quy trình định chuẩn:
1. Khởi tạo và thiết lập môi trường thư viện YOLOv11 tiên tiến nhất.
2. Kiểm định tài nguyên phần cứng (Khả năng đáp ứng của GPU Cuda).
3. Kích hoạt kỹ thuật Tích lũy Gradient (Gradient Accumulation) nhằm vượt qua rào cản VRAM vật lý, kết hợp cơ chế kiểm soát Overfitting.

In [1]:
# Khởi tạo thư viện cốt lõi
import sys
!{sys.executable} -m pip install -U ultralytics
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------------------------------- -------- 1.0/1.3 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 9.8 MB/s  0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.77
    Uninstalling ultralytics-8.4.77:
      Successfully uninstalled ultralytics-8.4.77
Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
# Kiểm định tính khả dụng của phần cứng GPU
import torch
if torch.cuda.is_available():
    print(f"Hệ thống nhận diện phần cứng xử lý: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Dung lượng VRAM khả dụng thực tế: {vram:.2f} GB")
else:
    print("Lỗi nghiêm trọng: Không phát hiện nhân tính toán CUDA. Quá trình phân tích sẽ bị gián đoạn.")

Hệ thống nhận diện phần cứng xử lý: NVIDIA GeForce RTX 3050 Laptop GPU
Dung lượng VRAM khả dụng thực tế: 4.29 GB


## Kích hoạt Mô hình YOLOv11 Nano (YOLO11n)
Áp dụng cấu hình phân rã khối lượng công việc nhằm bảo đảm an toàn VRAM (4GB):
- **Kích thước vật lý (Batch Size)** = 8 (Mức độ dung nạp tối đa của VRAM 4GB).
- **Hệ số giả lập Gradient tự động (Auto-Accumulate)**: Thư viện Ultralytics tự động thiết lập Accumulate = 64 / Batch. Với Batch=8, hệ thống sẽ ngầm định tích lũy ma trận sai số qua 8 chu kỳ trước khi tối ưu (8 x 8 = 64), giúp ổn định đạo hàm mà không cần code tay dài dòng.
- **Cơ chế Chống Overfitting**: Áp dụng Early Stopping (`patience=15`), kết hợp ngắt Augmentation (`close_mosaic=10`) ở các chu kỳ cuối nhằm tối ưu hóa độ sắc nét của viền đối tượng.

In [4]:
from ultralytics import YOLO
import shutil
import os

model = YOLO('yolo11n.pt')

results = model.train(
    data='d:/DAT301m/proposal/data/bdd_day.yaml',
    epochs=20,
    patience=15,           # Dừng sớm nếu sau 15 epoch mAP không tăng (Chống Overfit)
    batch=8,               # Kích thước khung nạp vật lý (Max VRAM 4GB)
    cache='disk',          # Bộ đệm ổ cứng giảm tải System RAM (~50GB SSD)
    imgsz=640,
    device=0,
    workers=4,             # Giảm từ 8 xuống 4 cho Windows (spawn overhead)
    optimizer='auto',      # Tự động chọn Optimizer + LR tối ưu theo kích thước dataset
    mosaic=0.5,            # Bật Data Augmentation ghép 4 ảnh
    close_mosaic=10,       # Tắt ghép ảnh ở 10 epoch cuối để học viền thực tế
    seed=0,                # Cố định hạt giống ngẫu nhiên (Tái tạo kết quả học thuật)
    project='d:/DAT301m/proposal/models/runs',
    name='bdd_day_train'
)

best_weight_path = 'd:/DAT301m/proposal/models/runs/bdd_day_train/weights/best.pt'
new_weight_path = 'd:/DAT301m/proposal/models/runs/bdd_day_train/weights/bdd_day_base_epoch20.pt'

if os.path.exists(best_weight_path):
    shutil.copy(best_weight_path, new_weight_path)
    print(f"\n Đã sao chép và đổi tên trọng số tốt nhất thành: {new_weight_path}")

Ultralytics 8.4.80  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:/DAT301m/proposal/data/bdd_day.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=bdd_day_train-2, nbs=64, nms=False, opset=None, optimize=False, optimiz